In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Текущая рабочая папка:', PROJECT_DIR)
data_json = PROJECT_DIR / 'data.json'
if data_json.exists():
    print('data.json найден')
else:
    print('data.json не найден — если запускаете ноутбук 01, положите его в текущую папку проекта:', PROJECT_DIR)
existing = sorted(p.name for p in ARTIFACT_DIR.iterdir())
if existing:
    print(f'В artifacts/ уже есть {len(existing)} файлов/папок:')
    for name in existing:
        print(' -', name)
else:
    print('Папка artifacts/ пока пустая, она заполнится после запуска обучающих ноутбуков.')

# 05 — Оптимизация RuBERT через Optuna и дополнительное сравнение xLSTM

Разделы:
8. Оптимизация гиперпараметров RuBERT: learning rate, dropout, weight decay  
9. Дополнительное сравнение архитектур: xLSTM-style модель  
10. Классический baseline для последовательных данных: bidirectional LSTM

Файл использует единое разбиение из `02_sentiment_split.joblib` и сохраняет метрики для итогового сравнения в файле `06`.

### Проверка зависимостей

In [2]:
import importlib.util
import subprocess
import sys
REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'datasets': 'datasets',
    'transformers': 'transformers',
    'torch': 'torch',
    'tqdm': 'tqdm',
    'joblib': 'joblib'}
missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Устанавливаются отсутствующие зависимости:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Все зависимости доступны.')
if importlib.util.find_spec('optuna') is None:
    print('Устанавливается optuna')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])

Все зависимости доступны.
Устанавливается optuna


## Импорты и конфигурация

In [3]:
import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_NAMES = ['Негативный', 'Позитивный', 'Нейтральный']
LABEL2NAME = {0: 'Негативный', 1: 'Позитивный', 2: 'Нейтральный'}
ID2LABEL = {0: 'NEGATIVE', 1: 'POSITIVE', 2: 'NEUTRAL'}
LABEL2ID = {'NEGATIVE': 0, 'POSITIVE': 1, 'NEUTRAL': 2}
TOXIC_KEYWORDS = ['депрессия', 'грусть', 'боль', 'слезы', 'слёзы', 'одиночество','апатия', 'ненависть', 'суицид', 'смерть', 'безысходность']
sns.set_theme(style='whitegrid', palette='Set2')
def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
set_seed(RANDOM_STATE)
print(f'Рабочая папка: {PROJECT_DIR}')
print(f'Папка артефактов: {ARTIFACT_DIR}')
import gc
from copy import deepcopy
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_STATE)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()} | Device: {DEVICE}')
import optuna

Рабочая папка: /content/drive/MyDrive/happiness_formula
Папка артефактов: /content/drive/MyDrive/happiness_formula/artifacts
PyTorch: 2.11.0+cu128
CUDA available: True | Device: cuda


In [4]:
def clean_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^а-яёА-ЯЁa-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

## 8. Оптимизация гиперпараметров RuBERT

In [7]:
RUBERT_MODEL_NAME = 'DeepPavlov/rubert-base-cased'
RUBERT_MAX_LENGTH = 192
RUBERT_BATCH_SIZE = 16 if DEVICE.type == 'cuda' else 8
class RuBERTSentimentDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.texts)
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
def make_rubert_loader(texts: List[str], labels: List[int], tokenizer, max_length: int, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = RuBERTSentimentDataset(texts, labels, tokenizer, max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
@torch.no_grad()
def evaluate_rubert_model(model: nn.Module, data_loader: DataLoader, loss_fn=None) -> Dict[str, object]:
    model.eval()
    losses, predictions, targets = [], [], []
    for batch in data_loader:
        labels = batch.pop('labels').to(DEVICE)
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        logits = model(**inputs).logits
        if loss_fn is not None:
            losses.append(loss_fn(logits, labels).item())
        predictions.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        targets.extend(labels.detach().cpu().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'f1_weighted': f1_score(targets, predictions, average='weighted'),
        'accuracy': accuracy_score(targets, predictions),
        'preds': predictions,
        'targets': targets}
def build_rubert_model(dropout: float, model_name: str = RUBERT_MODEL_NAME) -> nn.Module:
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True)
    return model.to(DEVICE)
def train_rubert_torch(
    train_texts: List[str],
    train_labels: List[int],
    valid_texts: List[str],
    valid_labels: List[int],
    *,
    learning_rate: float,
    dropout: float,
    weight_decay: float,
    epochs: int,
    batch_size: int,
    warmup_ratio: float = 0.10,
    use_amp: bool = True,
    trial: 'optuna.Trial' = None,
    checkpoint_path: Optional[Path] = None,
    resume: bool = True,
) -> Tuple[nn.Module, pd.DataFrame, Dict[str, object]]:
    set_seed(RANDOM_STATE)
    model = build_rubert_model(dropout=dropout)
    train_loader = make_rubert_loader(train_texts, train_labels, rubert_tokenizer, RUBERT_MAX_LENGTH, batch_size, True)
    valid_loader = make_rubert_loader(valid_texts, valid_labels, rubert_tokenizer, RUBERT_MAX_LENGTH, batch_size, False)
    class_counts = np.bincount(train_labels, minlength=3)
    class_weights = len(train_labels) / (3 * np.maximum(class_counts, 1))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    total_steps = max(1, len(train_loader) * epochs)
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    amp_enabled = use_amp and DEVICE.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    start_epoch = 1
    best_state = None
    best_valid_f1 = -1.0
    best_metrics = {}
    history = []
    if checkpoint_path is not None and resume and Path(checkpoint_path).exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_valid_f1 = checkpoint['best_valid_f1']
        best_metrics = checkpoint['best_metrics']
        best_state = checkpoint['best_state']
        history = checkpoint['history']
        print(f"Восстановлено из чекпоинта {checkpoint_path}: завершена эпоха "
            f"{checkpoint['epoch']}, продолжаем с эпохи {start_epoch}/{epochs}")
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f'RuBERT epoch {epoch}/{epochs}', leave=False):
            labels = batch.pop('labels').to(DEVICE)
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(**inputs).logits
                loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_losses.append(loss.item())
        valid_metrics = evaluate_rubert_model(model, valid_loader, loss_fn)
        row = {
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)),
            'valid_loss': valid_metrics['loss'],
            'valid_f1_weighted': valid_metrics['f1_weighted'],
            'valid_accuracy': valid_metrics['accuracy']}
        history.append(row)
        print(
            f"Epoch {epoch}: train_loss={row['train_loss']:.4f} | "
            f"valid_loss={row['valid_loss']:.4f} | "
            f"valid_f1={row['valid_f1_weighted']:.4f} | "
            f"valid_acc={row['valid_accuracy']:.4f}")
        if valid_metrics['f1_weighted'] > best_valid_f1:
            best_valid_f1 = valid_metrics['f1_weighted']
            best_metrics = valid_metrics
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if trial is not None:
            trial.report(valid_metrics['f1_weighted'], step=epoch)
            if trial.should_prune():
                del model
                gc.collect()
                if DEVICE.type == 'cuda':
                    torch.cuda.empty_cache()
                raise optuna.TrialPruned()
        if checkpoint_path is not None:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'best_valid_f1': best_valid_f1,
                'best_metrics': best_metrics,
                'best_state': best_state,
                'history': history,
            }, checkpoint_path)
            print(f'Чекпоинт сохранён: {checkpoint_path} (эпоха {epoch}/{epochs})')
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_metrics
@torch.no_grad()
def predict_rubert_sentiment(model: nn.Module, tokenizer, texts: List[str], batch_size: int = RUBERT_BATCH_SIZE) -> Tuple[List[int], List[float]]:
    if isinstance(texts, str):
        texts = [texts]
    loader = make_rubert_loader(texts, [0] * len(texts), tokenizer, RUBERT_MAX_LENGTH, batch_size, False)
    model.eval()
    predictions, confidences = [], []
    for batch in loader:
        batch.pop('labels')
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        probabilities = torch.softmax(model(**inputs).logits, dim=1)
        confidence, predicted = torch.max(probabilities, dim=1)
        predictions.extend(predicted.cpu().tolist())
        confidences.extend(confidence.cpu().tolist())
    return predictions, confidences

In [8]:
split_path = ARTIFACT_DIR / '02_sentiment_split.joblib'
if not split_path.exists():
    raise FileNotFoundError('Не найден artifacts/02_sentiment_split.joblib. Сначала запустите файл 02.')
split = joblib.load(split_path)
X_train_raw = split['X_train_raw']
X_test_raw = split['X_test_raw']
y_train = [int(value) for value in split['y_train']]
y_test = [int(value) for value in split['y_test']]
X_rubert_train, X_rubert_valid, y_rubert_train, y_rubert_valid = train_test_split(
    X_train_raw,
    y_train,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train)
rubert_tokenizer = AutoTokenizer.from_pretrained(RUBERT_MODEL_NAME)
print(f'RuBERT train: {len(X_rubert_train)}')
print(f'RuBERT valid: {len(X_rubert_valid)}')
print(f'RuBERT test:  {len(X_test_raw)}')
print(f'Batch size:   {RUBERT_BATCH_SIZE}')

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

RuBERT train: 5100
RuBERT valid: 900
RuBERT test:  1500
Batch size:   16


In [9]:
RUBERT_TUNING_EPOCHS = 2
RUBERT_OPTUNA_TRIALS = 4
RUBERT_OPTUNA_FINAL_EPOCHS = 4
DEFAULT_RUBERT_PARAMS = {
    'learning_rate': 2e-5,
    'dropout': 0.10,
    'weight_decay': 0.01}
optuna.logging.set_verbosity(optuna.logging.WARNING)
def rubert_objective(trial: optuna.Trial) -> float:
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True),
        'dropout': trial.suggest_float('dropout', 0.05, 0.30),
        'weight_decay': trial.suggest_float('weight_decay', 1e-4, 3e-2, log=True)}
    model, history, metrics = train_rubert_torch(
        X_rubert_train,
        y_rubert_train,
        X_rubert_valid,
        y_rubert_valid,
        learning_rate=params['learning_rate'],
        dropout=params['dropout'],
        weight_decay=params['weight_decay'],
        epochs=RUBERT_TUNING_EPOCHS,
        batch_size=RUBERT_BATCH_SIZE,
        trial=trial)
    score = float(metrics['f1_weighted'])
    trial.set_user_attr('best_valid_accuracy', float(metrics['accuracy']))
    trial.set_user_attr('history', history.to_dict('records'))
    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return score
rubert_pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=0)
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=rubert_pruner)
study.optimize(rubert_objective, n_trials=RUBERT_OPTUNA_TRIALS, show_progress_bar=True)
rubert_best_params = DEFAULT_RUBERT_PARAMS.copy()
rubert_best_params.update(study.best_params)
optuna_trials_df = study.trials_dataframe()
pruned_count = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f'Trial-ов отброшено pruner-ом: {pruned_count} из {len(study.trials)}')
display_columns = ['number', 'value', 'state', 'params_learning_rate', 'params_dropout', 'params_weight_decay']
display(optuna_trials_df[[column for column in display_columns if column in optuna_trials_df.columns]])
print('Лучшие гиперпараметры RuBERT:')
print(json.dumps(rubert_best_params, ensure_ascii=False, indent=2))

  0%|          | 0/4 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

Epoch 1: train_loss=0.8396 | valid_loss=0.7234 | valid_f1=0.6487 | valid_acc=0.6511


RuBERT epoch 2/2:   0%|          | 0/319 [00:00<?, ?it/s]

Epoch 2: train_loss=0.5486 | valid_loss=0.7377 | valid_f1=0.6906 | valid_acc=0.6900
Trial-ов отброшено pruner-ом: 1 из 4


,number,value,state,params_learning_rate,params_dropout,params_weight_decay
0,0,0.649560,COMPLETE,0.000018,0.287679,0.006505
1,1,0.688921,COMPLETE,0.000026,0.089005,0.000243
2,2,0.627220,PRUNED,0.000011,0.266544,0.003083
3,3,0.690634,COMPLETE,0.000031,0.055146,0.025269


Лучшие гиперпараметры RuBERT:
{
  "learning_rate": 3.12551431816761e-05,
  "dropout": 0.055146123573950614,
  "weight_decay": 0.025268782075084553
}


### 8.1 Финальное обучение RuBERT с лучшими параметрами Optuna

In [13]:
rubert_optuna_model, rubert_optuna_history, rubert_optuna_valid_metrics = train_rubert_torch(
    X_rubert_train,
    y_rubert_train,
    X_rubert_valid,
    y_rubert_valid,
    learning_rate=float(rubert_best_params['learning_rate']),
    dropout=float(rubert_best_params['dropout']),
    weight_decay=float(rubert_best_params['weight_decay']),
    epochs=RUBERT_OPTUNA_FINAL_EPOCHS,
    batch_size=RUBERT_BATCH_SIZE,
    checkpoint_path=ARTIFACT_DIR / '05_rubert_optuna_checkpoint.pt',
)
test_loader = make_rubert_loader(X_test_raw, y_test, rubert_tokenizer, RUBERT_MAX_LENGTH, RUBERT_BATCH_SIZE, False)
rubert_optuna_test_metrics = evaluate_rubert_model(rubert_optuna_model, test_loader)
rubert_optuna_preds = rubert_optuna_test_metrics['preds']
rubert_optuna_f1 = rubert_optuna_test_metrics['f1_weighted']
rubert_optuna_accuracy = rubert_optuna_test_metrics['accuracy']
print('Fine-tuned RuBERT, Optuna params')
print(classification_report(y_test, rubert_optuna_preds, target_names=TARGET_NAMES))
print(f'Weighted F1 RuBERT Optuna: {rubert_optuna_f1:.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

Восстановлено из чекпоинта /content/drive/MyDrive/happiness_formula/artifacts/05_rubert_optuna_checkpoint.pt: завершена эпоха 4, продолжаем с эпохи 5/4
Fine-tuned RuBERT, Optuna params
              precision    recall  f1-score   support

  Негативный       0.61      0.60      0.60       500
  Позитивный       0.83      0.77      0.80       500
 Нейтральный       0.68      0.75      0.71       500

    accuracy                           0.70      1500
   macro avg       0.71      0.70      0.70      1500
weighted avg       0.71      0.70      0.70      1500

Weighted F1 RuBERT Optuna: 0.7042


In [14]:
model_dir = ARTIFACT_DIR / '05_rubert_finetuned_optuna'
model_dir.mkdir(parents=True, exist_ok=True)
rubert_optuna_model.save_pretrained(model_dir)
rubert_tokenizer.save_pretrained(model_dir)
metrics = {
    'model': 'Fine-tuned RuBERT Optuna',
    'type': 'Transformer DL',
    'f1_weighted': float(rubert_optuna_f1),
    'accuracy': float(rubert_optuna_accuracy),
    'best_valid_f1_weighted': float(rubert_optuna_valid_metrics.get('f1_weighted', np.nan)),
    'best_valid_accuracy': float(rubert_optuna_valid_metrics.get('accuracy', np.nan)),
    'params': rubert_best_params,
    'tuning_epochs': RUBERT_TUNING_EPOCHS,
    'final_epochs': RUBERT_OPTUNA_FINAL_EPOCHS,
    'model_dir': str(model_dir)}
with open(ARTIFACT_DIR / '05_rubert_optuna_metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
optuna_trials_df.to_csv(ARTIFACT_DIR / '05_rubert_optuna_trials.csv', index=False, encoding='utf-8-sig')
rubert_optuna_history.to_csv(ARTIFACT_DIR / '05_rubert_optuna_history.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'text': X_test_raw, 'label_true': y_test, 'label_pred': rubert_optuna_preds}).to_csv(
    ARTIFACT_DIR / '05_rubert_optuna_predictions.csv', index=False, encoding='utf-8-sig')
print(f'Optuna RuBERT сохранён в: {model_dir}')
print(json.dumps(metrics, ensure_ascii=False, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Optuna RuBERT сохранён в: /content/drive/MyDrive/happiness_formula/artifacts/05_rubert_finetuned_optuna
{
  "model": "Fine-tuned RuBERT Optuna",
  "type": "Transformer DL",
  "f1_weighted": 0.7041670080240222,
  "accuracy": 0.7033333333333334,
  "best_valid_f1_weighted": 0.6990934942055168,
  "best_valid_accuracy": 0.6977777777777778,
  "params": {
    "learning_rate": 3.12551431816761e-05,
    "dropout": 0.055146123573950614,
    "weight_decay": 0.025268782075084553
  },
  "tuning_epochs": 2,
  "final_epochs": 4,
  "model_dir": "/content/drive/MyDrive/happiness_formula/artifacts/05_rubert_finetuned_optuna"
}


## 9. Дополнительное сравнение архитектур: xLSTM

In [15]:
RUN_XLSTM = True
XLSTM_MAX_LENGTH = 96
XLSTM_VOCAB_SIZE = 30_000
XLSTM_BATCH_SIZE = 64
XLSTM_EPOCHS = 4
XLSTM_EMBED_DIM = 128
XLSTM_HIDDEN_DIM = 128
XLSTM_DROPOUT = 0.30
def tokenize_for_xlstm(text: str) -> List[str]:
    return clean_text(text).split()
class Vocabulary:
    def __init__(self, max_size: int = XLSTM_VOCAB_SIZE, min_freq: int = 2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.token2id = {'<pad>': 0, '<unk>': 1}
    def fit(self, texts: List[str]) -> None:
        freq = {}
        for text in texts:
            for token in tokenize_for_xlstm(text):
                freq[token] = freq.get(token, 0) + 1
        sorted_tokens = sorted(freq.items(), key=lambda item: (-item[1], item[0]))
        for token, count in sorted_tokens:
            if count < self.min_freq or len(self.token2id) >= self.max_size:
                continue
            self.token2id[token] = len(self.token2id)
    def encode(self, text: str, max_length: int) -> List[int]:
        ids = [self.token2id.get(token, 1) for token in tokenize_for_xlstm(text)]
        ids = ids[:max_length]
        return ids + [0] * (max_length - len(ids))
    def __len__(self) -> int:
        return len(self.token2id)
    def to_dict(self) -> Dict[str, object]:
        return {'max_size': self.max_size, 'min_freq': self.min_freq, 'token2id': self.token2id}
    @classmethod
    def from_dict(cls, data: Dict[str, object]):
        vocab = cls(max_size=int(data.get('max_size', XLSTM_VOCAB_SIZE)), min_freq=int(data.get('min_freq', 2)))
        vocab.token2id = {str(key): int(value) for key, value in data['token2id'].items()}
        return vocab
class XLSTMDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], vocab: Vocabulary, max_length: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.texts)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        input_ids = self.vocab.encode(self.texts[idx], self.max_length)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
class sLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gates = nn.Linear(input_dim + hidden_dim, hidden_dim * 4)
    def forward(self, x_t: torch.Tensor, state: Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]):
        h_prev, c_prev, n_prev, m_prev = state
        z_t, i_t, f_t, o_t = self.gates(torch.cat([x_t, h_prev], dim=-1)).chunk(4, dim=-1)
        z_t = torch.tanh(z_t)
        o_t = torch.sigmoid(o_t)
        log_f_t = torch.nn.functional.logsigmoid(f_t)
        m_t = torch.maximum(log_f_t + m_prev, i_t)
        i_t_stable = torch.exp(i_t - m_t)
        f_t_stable = torch.exp(log_f_t + m_prev - m_t)
        c_t = f_t_stable * c_prev + i_t_stable * z_t
        n_t = f_t_stable * n_prev + i_t_stable
        h_t = o_t * (c_t / (n_t + 1e-6))
        return h_t, c_t, n_t, m_t
class XLSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, dropout: float, num_classes: int = 3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.cell = sLSTMCell(embed_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes))
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        embeddings = self.dropout(self.embedding(input_ids))
        batch_size, seq_len, _ = embeddings.shape
        h_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        c_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        n_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        m_t = torch.zeros(batch_size, self.hidden_dim, device=input_ids.device)
        outputs = []
        for step in range(seq_len):
            h_t, c_t, n_t, m_t = self.cell(embeddings[:, step, :], (h_t, c_t, n_t, m_t))
            outputs.append(h_t.unsqueeze(1))
        sequence_output = torch.cat(outputs, dim=1)
        lengths = (input_ids != 0).sum(dim=1).clamp(min=1)
        last_indices = (lengths - 1).view(-1, 1, 1).expand(-1, 1, self.hidden_dim)
        pooled = sequence_output.gather(dim=1, index=last_indices).squeeze(1)
        return self.classifier(pooled)
@torch.no_grad()
def evaluate_xlstm_model(model: nn.Module, data_loader: DataLoader, loss_fn=None) -> Dict[str, object]:
    model.eval()
    predictions, targets, losses = [], [], []
    for input_ids, labels in data_loader:
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(input_ids)
        if loss_fn is not None:
            losses.append(loss_fn(logits, labels).item())
        predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'f1_weighted': f1_score(targets, predictions, average='weighted'),
        'accuracy': accuracy_score(targets, predictions),
        'preds': predictions,
        'targets': targets}

In [16]:
if RUN_XLSTM:
    set_seed(RANDOM_STATE)
    xlstm_vocab = Vocabulary(max_size=XLSTM_VOCAB_SIZE, min_freq=2)
    xlstm_vocab.fit(X_train_raw)
    xlstm_train_loader = DataLoader(XLSTMDataset(X_rubert_train, y_rubert_train, xlstm_vocab, XLSTM_MAX_LENGTH), batch_size=XLSTM_BATCH_SIZE, shuffle=True)
    xlstm_valid_loader = DataLoader(XLSTMDataset(X_rubert_valid, y_rubert_valid, xlstm_vocab, XLSTM_MAX_LENGTH), batch_size=XLSTM_BATCH_SIZE, shuffle=False)
    xlstm_test_loader = DataLoader(XLSTMDataset(X_test_raw, y_test, xlstm_vocab, XLSTM_MAX_LENGTH), batch_size=XLSTM_BATCH_SIZE, shuffle=False)
    xlstm_model = XLSTMTextClassifier(
        vocab_size=len(xlstm_vocab),
        embed_dim=XLSTM_EMBED_DIM,
        hidden_dim=XLSTM_HIDDEN_DIM,
        dropout=XLSTM_DROPOUT).to(DEVICE)
    class_counts = np.bincount(y_rubert_train, minlength=3)
    class_weights = len(y_rubert_train) / (3 * np.maximum(class_counts, 1))
    xlstm_loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    xlstm_optimizer = torch.optim.AdamW(xlstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
    xlstm_best_state = None
    xlstm_best_f1 = -1.0
    xlstm_history = []
    for epoch in range(1, XLSTM_EPOCHS + 1):
        xlstm_model.train()
        train_losses = []
        for input_ids, labels in tqdm(xlstm_train_loader, desc=f'xLSTM epoch {epoch}/{XLSTM_EPOCHS}', leave=False):
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            xlstm_optimizer.zero_grad(set_to_none=True)
            logits = xlstm_model(input_ids)
            loss = xlstm_loss_fn(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(xlstm_model.parameters(), max_norm=1.0)
            xlstm_optimizer.step()
            train_losses.append(loss.item())
        valid_metrics = evaluate_xlstm_model(xlstm_model, xlstm_valid_loader, xlstm_loss_fn)
        xlstm_history.append({
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)),
            'valid_loss': valid_metrics['loss'],
            'valid_f1_weighted': valid_metrics['f1_weighted'],
            'valid_accuracy': valid_metrics['accuracy']})
        print(f"Epoch {epoch}: train_loss={np.mean(train_losses):.4f} | "
            f"valid_f1={valid_metrics['f1_weighted']:.4f} | valid_acc={valid_metrics['accuracy']:.4f}")
        if valid_metrics['f1_weighted'] > xlstm_best_f1:
            xlstm_best_f1 = valid_metrics['f1_weighted']
            xlstm_best_state = deepcopy(xlstm_model.state_dict())
    if xlstm_best_state is not None:
        xlstm_model.load_state_dict(xlstm_best_state)
    xlstm_test_metrics = evaluate_xlstm_model(xlstm_model, xlstm_test_loader)
    xlstm_preds = xlstm_test_metrics['preds']
    xlstm_f1 = xlstm_test_metrics['f1_weighted']
    xlstm_accuracy = xlstm_test_metrics['accuracy']
    print('xLSTM-style classifier')
    print(classification_report(y_test, xlstm_preds, target_names=TARGET_NAMES))
else:
    xlstm_model = None
    xlstm_vocab = None
    xlstm_history = []
    xlstm_preds = []
    xlstm_f1 = np.nan
    xlstm_accuracy = np.nan

xLSTM epoch 1/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 1: train_loss=1.1398 | valid_f1=0.4319 | valid_acc=0.4300


xLSTM epoch 2/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 2: train_loss=1.0221 | valid_f1=0.4620 | valid_acc=0.4856


xLSTM epoch 3/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 3: train_loss=0.9227 | valid_f1=0.5098 | valid_acc=0.5244


xLSTM epoch 4/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 4: train_loss=0.8371 | valid_f1=0.4990 | valid_acc=0.5211
xLSTM-style classifier
              precision    recall  f1-score   support

  Негативный       0.41      0.26      0.32       500
  Позитивный       0.54      0.71      0.61       500
 Нейтральный       0.48      0.49      0.49       500

    accuracy                           0.49      1500
   macro avg       0.48      0.49      0.47      1500
weighted avg       0.48      0.49      0.47      1500



In [17]:
if RUN_XLSTM:
    xlstm_metrics = {
        'model': 'xLSTM-style',
        'type': 'Recurrent DL',
        'f1_weighted': float(xlstm_f1),
        'accuracy': float(xlstm_accuracy),
        'epochs': XLSTM_EPOCHS,
        'max_length': XLSTM_MAX_LENGTH,
        'vocab_size': len(xlstm_vocab),
        'embed_dim': XLSTM_EMBED_DIM,
        'hidden_dim': XLSTM_HIDDEN_DIM,
        'dropout': XLSTM_DROPOUT,
        'state_dict_path': str(ARTIFACT_DIR / '05_xlstm_state.pt'),
        'vocab_path': str(ARTIFACT_DIR / '05_xlstm_vocab.json')}
    torch.save(xlstm_model.state_dict(), ARTIFACT_DIR / '05_xlstm_state.pt')
    with open(ARTIFACT_DIR / '05_xlstm_vocab.json', 'w', encoding='utf-8') as file:
        json.dump(xlstm_vocab.to_dict(), file, ensure_ascii=False)
    with open(ARTIFACT_DIR / '05_xlstm_metrics.json', 'w', encoding='utf-8') as file:
        json.dump(xlstm_metrics, file, ensure_ascii=False, indent=2)
    pd.DataFrame(xlstm_history).to_csv(ARTIFACT_DIR / '05_xlstm_history.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame({'text': X_test_raw, 'label_true': y_test, 'label_pred': xlstm_preds}).to_csv(
        ARTIFACT_DIR / '05_xlstm_predictions.csv', index=False, encoding='utf-8-sig')
    print('Метрики xLSTM:')
    print(json.dumps(xlstm_metrics, ensure_ascii=False, indent=2))

Метрики xLSTM:
{
  "model": "xLSTM-style",
  "type": "Recurrent DL",
  "f1_weighted": 0.473329782992059,
  "accuracy": 0.49,
  "epochs": 4,
  "max_length": 96,
  "vocab_size": 19623,
  "embed_dim": 128,
  "hidden_dim": 128,
  "dropout": 0.3,
  "state_dict_path": "/content/drive/MyDrive/happiness_formula/artifacts/05_xlstm_state.pt",
  "vocab_path": "/content/drive/MyDrive/happiness_formula/artifacts/05_xlstm_vocab.json"
}


## 10. Классическая архитектура: LSTM (baseline для последовательных данных)


In [18]:
LSTM_MAX_LENGTH = 96
LSTM_BATCH_SIZE = 64
LSTM_EPOCHS = 4
LSTM_EMBED_DIM = 128
LSTM_HIDDEN_DIM = 128
LSTM_NUM_LAYERS = 1
LSTM_DROPOUT = 0.30
LSTM_BIDIRECTIONAL = True
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int, dropout: float, bidirectional: bool, num_classes: int = 3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0)
        output_dim = hidden_dim * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
            nn.Linear(output_dim, num_classes))
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        lengths = (input_ids != 0).sum(dim=1).clamp(min=1).cpu()
        embeddings = self.dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        if self.lstm.bidirectional:
            pooled = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        else:
            pooled = h_n[-1]
        return self.classifier(pooled)
@torch.no_grad()
def evaluate_lstm_model(model: nn.Module, data_loader: DataLoader, loss_fn=None) -> Dict[str, object]:
    model.eval()
    predictions, targets, losses = [], [], []
    for input_ids, labels in data_loader:
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(input_ids)
        if loss_fn is not None:
            losses.append(loss_fn(logits, labels).item())
        predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'f1_weighted': f1_score(targets, predictions, average='weighted'),
        'accuracy': accuracy_score(targets, predictions),
        'preds': predictions,
        'targets': targets}

In [19]:
if RUN_XLSTM:
    set_seed(RANDOM_STATE)
    lstm_model = LSTMTextClassifier(
        vocab_size=len(xlstm_vocab),
        embed_dim=LSTM_EMBED_DIM,
        hidden_dim=LSTM_HIDDEN_DIM,
        num_layers=LSTM_NUM_LAYERS,
        dropout=LSTM_DROPOUT,
        bidirectional=LSTM_BIDIRECTIONAL).to(DEVICE)
    lstm_loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    lstm_optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
    lstm_best_state = None
    lstm_best_f1 = -1.0
    lstm_history = []
    for epoch in range(1, LSTM_EPOCHS + 1):
        lstm_model.train()
        train_losses = []
        for input_ids, labels in tqdm(xlstm_train_loader, desc=f'LSTM epoch {epoch}/{LSTM_EPOCHS}', leave=False):
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            lstm_optimizer.zero_grad(set_to_none=True)
            logits = lstm_model(input_ids)
            loss = lstm_loss_fn(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)
            lstm_optimizer.step()
            train_losses.append(loss.item())
        valid_metrics = evaluate_lstm_model(lstm_model, xlstm_valid_loader, lstm_loss_fn)
        lstm_history.append({
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)),
            'valid_loss': valid_metrics['loss'],
            'valid_f1_weighted': valid_metrics['f1_weighted'],
            'valid_accuracy': valid_metrics['accuracy']})
        print(f"Epoch {epoch}: train_loss={np.mean(train_losses):.4f} | "
            f"valid_f1={valid_metrics['f1_weighted']:.4f} | valid_acc={valid_metrics['accuracy']:.4f}")
        if valid_metrics['f1_weighted'] > lstm_best_f1:
            lstm_best_f1 = valid_metrics['f1_weighted']
            lstm_best_state = deepcopy(lstm_model.state_dict())
    if lstm_best_state is not None:
        lstm_model.load_state_dict(lstm_best_state)
    lstm_test_metrics = evaluate_lstm_model(lstm_model, xlstm_test_loader)
    lstm_preds = lstm_test_metrics['preds']
    lstm_f1 = lstm_test_metrics['f1_weighted']
    lstm_accuracy = lstm_test_metrics['accuracy']
    print('Classic LSTM classifier')
    print(classification_report(y_test, lstm_preds, target_names=TARGET_NAMES))
else:
    lstm_model = None
    lstm_history = []
    lstm_preds = []
    lstm_f1 = np.nan
    lstm_accuracy = np.nan

LSTM epoch 1/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 1: train_loss=1.1128 | valid_f1=0.4986 | valid_acc=0.5011


LSTM epoch 2/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 2: train_loss=0.9425 | valid_f1=0.5298 | valid_acc=0.5289


LSTM epoch 3/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 3: train_loss=0.8574 | valid_f1=0.5433 | valid_acc=0.5533


LSTM epoch 4/4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 4: train_loss=0.7651 | valid_f1=0.5544 | valid_acc=0.5578
Classic LSTM classifier
              precision    recall  f1-score   support

  Негативный       0.47      0.43      0.45       500
  Позитивный       0.61      0.64      0.62       500
 Нейтральный       0.53      0.54      0.53       500

    accuracy                           0.54      1500
   macro avg       0.53      0.54      0.54      1500
weighted avg       0.53      0.54      0.54      1500



In [20]:
if RUN_XLSTM:
    lstm_metrics = {
        'model': 'Classic LSTM',
        'type': 'Recurrent DL',
        'f1_weighted': float(lstm_f1),
        'accuracy': float(lstm_accuracy),
        'epochs': LSTM_EPOCHS,
        'max_length': LSTM_MAX_LENGTH,
        'vocab_size': len(xlstm_vocab),
        'embed_dim': LSTM_EMBED_DIM,
        'hidden_dim': LSTM_HIDDEN_DIM,
        'num_layers': LSTM_NUM_LAYERS,
        'bidirectional': LSTM_BIDIRECTIONAL,
        'dropout': LSTM_DROPOUT,
        'state_dict_path': str(ARTIFACT_DIR / '05_lstm_state.pt'),
        'vocab_path': str(ARTIFACT_DIR / '05_xlstm_vocab.json')}
    torch.save(lstm_model.state_dict(), ARTIFACT_DIR / '05_lstm_state.pt')
    with open(ARTIFACT_DIR / '05_lstm_metrics.json', 'w', encoding='utf-8') as file:
        json.dump(lstm_metrics, file, ensure_ascii=False, indent=2)
    pd.DataFrame(lstm_history).to_csv(ARTIFACT_DIR / '05_lstm_history.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame({'text': X_test_raw, 'label_true': y_test, 'label_pred': lstm_preds}).to_csv(
        ARTIFACT_DIR / '05_lstm_predictions.csv', index=False, encoding='utf-8-sig')
    print('Метрики Classic LSTM:')
    print(json.dumps(lstm_metrics, ensure_ascii=False, indent=2))

Метрики Classic LSTM:
{
  "model": "Classic LSTM",
  "type": "Recurrent DL",
  "f1_weighted": 0.535353246942259,
  "accuracy": 0.5373333333333333,
  "epochs": 4,
  "max_length": 96,
  "vocab_size": 19623,
  "embed_dim": 128,
  "hidden_dim": 128,
  "num_layers": 1,
  "bidirectional": true,
  "dropout": 0.3,
  "state_dict_path": "/content/drive/MyDrive/happiness_formula/artifacts/05_lstm_state.pt",
  "vocab_path": "/content/drive/MyDrive/happiness_formula/artifacts/05_xlstm_vocab.json"
}
